# Experimento 02 — Logistic Regression

Compara V1–V4 y realiza un ajuste moderado de `C` y `class_weight` sobre la mejor variante base.

Sólo se utilizan train y validation. Test no se carga.

In [1]:
from pathlib import Path
import json,time
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix,f1_score,precision_score,recall_score
ROOT=Path.cwd().parent if Path.cwd().name=="experimentos" else Path.cwd()
DATA=ROOT/"data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet"
SPLITS=ROOT/"data/processed/modelado/splits";OUT=ROOT/"results/experimentos_parciales";OUT.mkdir(parents=True,exist_ok=True)
datos=pd.read_parquet(DATA);datos.fecha_inicio_semana=pd.to_datetime(datos.fecha_inicio_semana)
KEY=["departamento","fecha_inicio_semana"];TARGET="nivel_actividad_firms";ORDER=["Sin actividad","Bajo","Moderado","Alto"];RS=42
def subset(name):
 k=pd.read_parquet(SPLITS/f"{name}_keys.parquet");k.fecha_inicio_semana=pd.to_datetime(k.fecha_inicio_semana)
 return datos.merge(k,on=KEY,how="inner",validate="one_to_one")
train=subset("train");validation=subset("validation")
assert len(train)==5546 and len(validation)==1188
CAT=["departamento","mes"]
MET=["temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]
ROLL=["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
VARIANTS={"V1":CAT+MET,"V2":CAT+MET+LAGS,"V3":CAT+MET+ROLL,"V4":CAT+MET+LAGS+ROLL}
PROHIBITED={"cantidad_detecciones","fecha_inicio_semana","anio","numero_semana",TARGET,"nivel_actividad_firms_codigo"}
assert all(PROHIBITED.isdisjoint(v) for v in VARIANTS.values())
def metrics(y,p):
 r=classification_report(y,p,labels=ORDER,output_dict=True,zero_division=0)
 return {"accuracy_validation":accuracy_score(y,p),"balanced_accuracy_validation":balanced_accuracy_score(y,p),"precision_macro_validation":precision_score(y,p,average="macro",zero_division=0),"recall_macro_validation":recall_score(y,p,average="macro",zero_division=0),"f1_macro_validation":f1_score(y,p,average="macro",zero_division=0),"f1_weighted_validation":f1_score(y,p,average="weighted",zero_division=0),**{f"precision_{x.lower().replace(' ','_')}":r[x]["precision"] for x in ORDER},**{f"recall_{x.lower().replace(' ','_')}":r[x]["recall"] for x in ORDER},**{f"f1_{x.lower().replace(' ','_')}":r[x]["f1-score"] for x in ORDER}}
def row(eid,model,variant,params,pred,seconds,note):return {"experiment_id":eid,"modelo":model,"variante_variables":variant,"hiperparametros":json.dumps(params,sort_keys=True,ensure_ascii=False),**metrics(validation[TARGET],pred),"duracion_segundos":seconds,"fecha_ejecucion":datetime.now(timezone.utc).isoformat(),"notebook_origen":NOTEBOOK,"observaciones":note}

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
NOTEBOOK="Experimento_02_LogisticRegression.ipynb"
def build(cols,C=1.0,class_weight="balanced"):
 cat=[x for x in cols if x in CAT];num=[x for x in cols if x not in CAT]
 pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),cat),("num",StandardScaler(),num)])
 return Pipeline([("pre",pre),("model",LogisticRegression(C=C,max_iter=2000,class_weight=class_weight,random_state=RS))])

In [2]:
rows=[];base_models={}
for v,cols in VARIANTS.items():
 t=time.perf_counter();mod=build(cols).fit(train[cols],train[TARGET]);pred=mod.predict(validation[cols]);elapsed=time.perf_counter()-t;base_models[v]=mod
 rows.append(row(f"lr_base_{v.lower()}","LogisticRegression",v,{"C":1.0,"class_weight":"balanced","max_iter":2000,"random_state":RS},pred,elapsed,"Configuración base"))
base=pd.DataFrame(rows);base[["variante_variables","f1_macro_validation","balanced_accuracy_validation","f1_bajo","f1_moderado","f1_alto","duracion_segundos"]].round(4)
for v,mod in base_models.items():
 print(f"Matriz de confusión validation - {v}")
 pred_v=mod.predict(validation[VARIANTS[v]])
 display(pd.DataFrame(confusion_matrix(validation[TARGET],pred_v,labels=ORDER),index=ORDER,columns=ORDER))

Matriz de confusión validation - V1


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,416,129,87,113
Bajo,49,51,40,45
Moderado,44,39,47,60
Alto,8,7,13,40


Matriz de confusión validation - V2


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,443,131,99,72
Bajo,58,42,42,43
Moderado,52,51,43,44
Alto,7,8,13,40


Matriz de confusión validation - V3


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,436,133,96,80
Bajo,55,42,46,42
Moderado,50,41,51,48
Alto,8,8,14,38


Matriz de confusión validation - V4


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,438,131,102,74
Bajo,57,40,46,42
Moderado,50,43,49,48
Alto,8,8,12,40


In [3]:
best_variant=base.sort_values(["f1_macro_validation","balanced_accuracy_validation"],ascending=False).iloc[0].variante_variables
rows_tuned=[]
for C in [0.1,1.0,10.0]:
 for cw in [None,"balanced"]:
  cols=VARIANTS[best_variant];t=time.perf_counter();mod=build(cols,C,cw).fit(train[cols],train[TARGET]);pred=mod.predict(validation[cols]);elapsed=time.perf_counter()-t
  rows_tuned.append(row(f"lr_tuned_{best_variant.lower()}_c{C}_cw{cw}","LogisticRegression",best_variant,{"C":C,"class_weight":cw,"max_iter":2000,"random_state":RS},pred,elapsed,"Ajuste moderado sobre mejor variante base"))
results=pd.concat([base,pd.DataFrame(rows_tuned)],ignore_index=True);results.sort_values("f1_macro_validation",ascending=False).head(10).round(4)

,experiment_id,modelo,variante_variables,hiperparametros,accuracy_validation,balanced_accuracy_validation,precision_macro_validation,recall_macro_validation,f1_macro_validation,f1_weighted_validation,...,recall_moderado,recall_alto,f1_sin_actividad,f1_bajo,f1_moderado,f1_alto,duracion_segundos,fecha_ejecucion,notebook_origen,observaciones
5,lr_tuned_v3_c0.1_cwbalanced,LogisticRegression,V3,"{""C"": 0.1, ""class_weight"": ""balanced"", ""max_it...",0.4907,0.4229,0.3601,0.4229,0.3641,0.5201,...,0.2737,0.5882,0.6819,0.2084,0.2687,0.2974,0.1468,2026-09-30T19:15:06.588533+00:00,Experimento_02_LogisticRegression.ipynb,Ajuste moderado sobre mejor variante base
9,lr_tuned_v3_c10.0_cwbalanced,LogisticRegression,V3,"{""C"": 10.0, ""class_weight"": ""balanced"", ""max_i...",0.4781,0.4102,0.3532,0.4102,0.3538,0.5121,...,0.2684,0.5588,0.6749,0.2034,0.2576,0.2794,0.1847,2026-09-30T19:15:07.774599+00:00,Experimento_02_LogisticRegression.ipynb,Ajuste moderado sobre mejor variante base
7,lr_tuned_v3_c1.0_cwbalanced,LogisticRegression,V3,"{""C"": 1.0, ""class_weight"": ""balanced"", ""max_it...",0.4773,0.4099,0.3527,0.4099,0.3529,0.5114,...,0.2684,0.5588,0.6739,0.2054,0.2569,0.2754,0.1852,2026-09-30T19:15:07.186065+00:00,Experimento_02_LogisticRegression.ipynb,Ajuste moderado sobre mejor variante base
2,lr_base_v3,LogisticRegression,V3,"{""C"": 1.0, ""class_weight"": ""balanced"", ""max_it...",0.4773,0.4099,0.3527,0.4099,0.3529,0.5114,...,0.2684,0.5588,0.6739,0.2054,0.2569,0.2754,0.2087,2026-09-30T19:15:04.565252+00:00,Experimento_02_LogisticRegression.ipynb,Configuración base
3,lr_base_v4,LogisticRegression,V4,"{""C"": 1.0, ""class_weight"": ""balanced"", ""max_it...",0.4773,0.4126,0.3507,0.4126,0.3528,0.5099,...,0.2579,0.5882,0.6749,0.1966,0.2456,0.2941,1.3702,2026-09-30T19:15:06.031407+00:00,Experimento_02_LogisticRegression.ipynb,Configuración base
1,lr_base_v2,LogisticRegression,V2,"{""C"": 1.0, ""class_weight"": ""balanced"", ""max_it...",0.4781,0.4091,0.3478,0.4091,0.3506,0.5098,...,0.2263,0.5882,0.6789,0.2014,0.2222,0.2996,0.1477,2026-09-30T19:15:04.262721+00:00,Experimento_02_LogisticRegression.ipynb,Configuración base
0,lr_base_v1,LogisticRegression,V1,"{""C"": 1.0, ""class_weight"": ""balanced"", ""max_it...",0.4663,0.4174,0.3592,0.4174,0.3505,0.5060,...,0.2474,0.5882,0.6593,0.2482,0.2493,0.2454,0.1873,2026-09-30T19:15:04.020504+00:00,Experimento_02_LogisticRegression.ipynb,Configuración base
6,lr_tuned_v3_c1.0_cwNone,LogisticRegression,V3,"{""C"": 1.0, ""class_weight"": null, ""max_iter"": 2...",0.6254,0.3080,0.3147,0.3080,0.2899,0.5268,...,0.1000,0.1765,0.7824,0.0000,0.1418,0.2353,0.2185,2026-09-30T19:15:06.906796+00:00,Experimento_02_LogisticRegression.ipynb,Ajuste moderado sobre mejor variante base
8,lr_tuned_v3_c10.0_cwNone,LogisticRegression,V3,"{""C"": 10.0, ""class_weight"": null, ""max_iter"": ...",0.6237,0.3074,0.3082,0.3074,0.2880,0.5257,...,0.1000,0.1765,0.7811,0.0000,0.1423,0.2286,0.2165,2026-09-30T19:15:07.496400+00:00,Experimento_02_LogisticRegression.ipynb,Ajuste moderado sobre mejor variante base
4,lr_tuned_v3_c0.1_cwNone,LogisticRegression,V3,"{""C"": 0.1, ""class_weight"": null, ""max_iter"": 2...",0.6221,0.2861,0.2956,0.2861,0.2626,0.5165,...,0.0789,0.1029,0.7802,0.0000,0.1145,0.1556,0.1411,2026-09-30T19:15:06.347154+00:00,Experimento_02_LogisticRegression.ipynb,Ajuste moderado sobre mejor variante base


In [4]:
best=results.sort_values(["f1_macro_validation","balanced_accuracy_validation"],ascending=False).iloc[0];params=json.loads(best.hiperparametros);cols=VARIANTS[best.variante_variables];final=build(cols,params["C"],params["class_weight"]).fit(train[cols],train[TARGET]);pred=final.predict(validation[cols]);display(pd.DataFrame(confusion_matrix(validation[TARGET],pred,labels=ORDER),index=ORDER,columns=ORDER));display(pd.DataFrame(classification_report(validation[TARGET],pred,labels=ORDER,output_dict=True,zero_division=0)).T.loc[ORDER].round(4));print(best[["experiment_id","variante_variables","hiperparametros"]])

,Sin actividad,Bajo,Moderado,Alto
Sin actividad,449,131,88,77
Bajo,58,42,45,40
Moderado,56,38,52,44
Alto,9,7,12,40


,precision,recall,f1-score,support
Sin actividad,0.7850,0.6027,0.6819,745.0
Bajo,0.1927,0.2270,0.2084,185.0
Moderado,0.2640,0.2737,0.2687,190.0
Alto,0.1990,0.5882,0.2974,68.0


experiment_id                               lr_tuned_v3_c0.1_cwbalanced
variante_variables                                                   V3
hiperparametros       {"C": 0.1, "class_weight": "balanced", "max_it...
Name: 5, dtype: object


In [5]:
coef=pd.DataFrame(final.named_steps["model"].coef_,index=final.named_steps["model"].classes_,columns=final.named_steps["pre"].get_feature_names_out());
for cl in ORDER: print("\n",cl);display(coef.loc[cl].sort_values(key=abs,ascending=False).head(5).to_frame("coeficiente").round(4))


 Sin actividad


,coeficiente
cat__mes_8,-0.7518
cat__mes_2,0.6594
cat__departamento_Flores,0.6481
cat__departamento_Salto,0.6008
cat__mes_3,0.5428



 Bajo


,coeficiente
cat__departamento_Canelones,-0.5255
cat__departamento_Flores,0.4242
cat__mes_11,0.4202
cat__departamento_Durazno,0.2962
cat__mes_12,0.2767



 Moderado


,coeficiente
cat__departamento_Florida,0.3621
cat__departamento_Flores,-0.3603
cat__mes_11,0.3242
cat__mes_10,0.2802
cat__mes_3,-0.2620



 Alto


,coeficiente
cat__mes_11,-0.8506
cat__mes_8,0.7857
cat__departamento_Salto,-0.7730
cat__departamento_Flores,-0.7120
cat__departamento_Florida,-0.6429


In [6]:
results.to_csv(OUT/"02_logistic_regression.csv",index=False);print("Guardado")

Guardado
